# V11 v2 — Unsupervised Clustering (matching Helene's V10.1 cycle extraction)

Key fixes vs V11 v1:
1. Fixed 64-sample window centered on peak (no resampling)
2. Peak pairing by closest timestamp within 0.25s
3. Omega in deg/s (not rad/s)
4. Double Savitzky-Golay smoothing
5. PCA capped at 50 components
6. K-means with n_init=20, max_iter=500

In [ ]:
# ── Cell 1: Setup ────────────────────────────────────────────
!git clone https://github.com/helenejabbour/ropeflow-project.git 2>/dev/null || echo 'Already cloned'
!pip install hdbscan umap-learn -q

import os
BASE = 'ropeflow-project'
DATA_PROCESSED = os.path.join(BASE, 'data', 'processed')
NEW_LABELED_RAW = os.path.join(BASE, 'data', 'raw', 'new-labeled-sessions')
RESULTS_DIR = 'v11_v2_results'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Setup done')

In [ ]:
# ── Cell 2: Imports ───────────────────────────────────────────
import glob, re, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter, find_peaks
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score
from sklearn.manifold import TSNE
import hdbscan
import umap
from collections import Counter, defaultdict
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print('Imports OK')

In [ ]:
# ── Cell 3: Cycle extraction functions (matching Helene's V10.1) ──

CONFIG = {
    'FS': 50.0,
    'PEAK_PROM_DEGS': 100.0,         # prominence threshold in deg/s
    'PEAK_MIN_DEGS': 50.0,            # reject peaks below this magnitude
    'PEAK_SAVGOL_WINDOW': 21,          # smoothing window (odd number)
    'PEAK_SAVGOL_POLY': 3,             # polynomial order
    'PEAK_MIN_PERIOD_S': 0.5,          # min distance between peaks
    'PEAK_PAIR_MAX_DT_S': 0.25,        # max time diff for L/R peak pairing
    'WINDOW': 64,                      # fixed window size around each peak
}

def extract_signals(df):
    t = df['timestamp_ms'].values / 1000.0
    A = df[['ax_w', 'ay_w', 'az_w']].values
    omega = df[['gx', 'gy', 'gz']].values * (np.pi / 180.0)  # rad/s internally
    return t, A, omega

def _smooth_mag_deg(omega_rad, cfg):
    """Double Savitzky-Golay smoothing on ||omega|| in deg/s."""
    mag_deg = np.linalg.norm(omega_rad, axis=1) * (180.0 / np.pi)
    n = len(mag_deg)
    if n < 7:
        return mag_deg
    win = int(cfg['PEAK_SAVGOL_WINDOW'])
    if win % 2 == 0:
        win += 1
    max_odd = n if n % 2 == 1 else n - 1
    win = max(5, min(win, max_odd))
    poly = max(1, min(int(cfg['PEAK_SAVGOL_POLY']), win - 2))
    # Two-pass smoothing for cleaner detection
    y = savgol_filter(mag_deg, window_length=win, polyorder=poly, mode='interp')
    y = savgol_filter(y, window_length=win, polyorder=poly, mode='interp')
    return y

def detect_cycle_peaks(omega_rad, fs, cfg):
    """Detect peaks in smoothed ||omega|| (deg/s). Returns peak indices."""
    mag_smooth = _smooth_mag_deg(omega_rad, cfg)
    peaks, _ = find_peaks(
        mag_smooth,
        distance=max(1, int(cfg['PEAK_MIN_PERIOD_S'] * fs)),
        prominence=cfg['PEAK_PROM_DEGS'],
    )
    # Reject peaks below minimum magnitude
    peaks = np.array([int(p) for p in peaks if mag_smooth[p] >= cfg['PEAK_MIN_DEGS']], dtype=int)
    return peaks, mag_smooth

def pair_peaks_same_swing(t0, peaks0, t1, peaks1, max_dt_s):
    """Pair L/R peaks by closest timestamp within max_dt_s."""
    if len(peaks0) == 0 or len(peaks1) == 0:
        return []
    used, pairs = set(), []
    t1_peaks = t1[peaks1]
    for p0 in peaks0:
        d = np.abs(t1_peaks - t0[p0])
        for idx in np.argsort(d):
            p1 = int(peaks1[idx])
            if p1 in used:
                continue
            if d[idx] <= max_dt_s:
                used.add(p1)
                pairs.append((int(p0), p1))
                break
    return pairs

def extract_fixed_window(ch6, center_idx, window=64):
    """Extract fixed-size window centered on peak. No resampling."""
    half = window // 2
    start = int(center_idx) - half
    end = start + window
    out = np.zeros((6, window), dtype=np.float32)
    src_lo = max(0, start)
    src_hi = min(ch6.shape[0], end)
    if src_hi <= src_lo:
        return out
    dst_lo = src_lo - start
    out[:, dst_lo:dst_lo + (src_hi - src_lo)] = ch6[src_lo:src_hi].T
    return out

# Label mapping
KNOWN_PATTERNS = {'overhand', 'sneak_overhand', 'underhand', 'sneak_underhand',
                  'dragon_roll', 'underhand_default'}
_EXACT = {
    'underhand': 'underhand', 'overhand': 'overhand', 'dragon_roll': 'dragon_roll',
    'sneak_underhand': 'sneak_underhand', 'sneak_overhand': 'sneak_overhand', 'idle': None,
}
_PREFIX_RULES = [
    (re.compile(r'^us', re.I), 'sneak_underhand'), (re.compile(r'^os', re.I), 'sneak_overhand'),
    (re.compile(r'^u', re.I), 'underhand'), (re.compile(r'^o', re.I), 'overhand'),
    (re.compile(r'^fb', re.I), 'dragon_roll'), (re.compile(r'^bf', re.I), 'dragon_roll'),
    (re.compile(r'^cw$', re.I), 'CW'), (re.compile(r'^ccw$', re.I), 'CCW'),
    (re.compile(r'^idle', re.I), None), (re.compile(r'^vq', re.I), None),
]
def map_label(raw):
    raw = raw.strip()
    if raw.lower() in _EXACT: return _EXACT[raw.lower()]
    for pat, c in _PREFIX_RULES:
        if pat.match(raw): return c
    return None

print('Functions defined')

In [ ]:
# ── Cell 4: Load ALL cycles using V10.1's extraction method ──

all_matrices = []      # (12, 64) each
all_labels = []
all_session_ids = []
session_names = []

def process_session_v10(d0_path, d1_path, session_name, label='unlabeled', time_windows=None):
    """Extract cycles using V10.1 method: fixed window around peaks, omega in deg/s."""
    df0 = pd.read_csv(d0_path)
    df1 = pd.read_csv(d1_path)
    t0, A0, om0 = extract_signals(df0)
    t1, A1, om1 = extract_signals(df1)
    fs = CONFIG['FS']

    # Detect peaks (double-smoothed, in deg/s)
    peaks0, mag0 = detect_cycle_peaks(om0, fs, CONFIG)
    peaks1, mag1 = detect_cycle_peaks(om1, fs, CONFIG)

    # Pair by closest timestamp within 0.25s
    pairs = pair_peaks_same_swing(t0, peaks0, t1, peaks1, CONFIG['PEAK_PAIR_MAX_DT_S'])

    # Build channels in deg/s (not rad/s) — matching Helene's approach
    ch0 = np.column_stack([A0, om0 * (180.0 / np.pi)])  # (N, 6) with omega in deg/s
    ch1 = np.column_stack([A1, om1 * (180.0 / np.pi)])

    sid = len(session_names)
    session_names.append(session_name)
    count = 0
    for p0, p1 in pairs:
        t_mid = 0.5 * (t0[p0] + t1[p1])
        # Filter by time windows if heterogeneous
        if time_windows is not None:
            if not any(ws <= t_mid < we for ws, we in time_windows):
                continue
        # Fixed 64-sample window centered on peak — NO resampling
        w0 = extract_fixed_window(ch0, p0, CONFIG['WINDOW'])  # (6, 64)
        w1 = extract_fixed_window(ch1, p1, CONFIG['WINDOW'])  # (6, 64)
        cm = np.vstack([w0, w1]).astype(np.float32)            # (12, 64)
        all_matrices.append(cm)
        all_labels.append(label)
        all_session_ids.append(sid)
        count += 1
    return count

# ── Homogeneous sessions ──
d0_files = sorted(glob.glob(os.path.join(DATA_PROCESSED, '*_device0_processed.csv')))
for d0 in d0_files:
    d1 = d0.replace('_device0_', '_device1_')
    if not os.path.exists(d1): continue
    stem = os.path.basename(d0).replace('_device0_processed.csv', '')
    parts = stem.split('_')
    if len(parts) < 3: continue
    rest = '_'.join(parts[2:])
    label = 'unlabeled'
    for pat in sorted(KNOWN_PATTERNS, key=len, reverse=True):
        if rest.startswith(pat):
            label = 'underhand' if pat == 'underhand_default' else pat
            break
    n = process_session_v10(d0, d1, stem, label)
    print(f'  {stem:<55s} {label:<20s} {n:>4d} cycles')

# ── Heterogeneous sessions ──
if os.path.isdir(NEW_LABELED_RAW):
    for sname in sorted(os.listdir(NEW_LABELED_RAW)):
        sdir = os.path.join(NEW_LABELED_RAW, sname)
        if not os.path.isdir(sdir): continue
        lpath = None
        for fn in ('labels_corrected.json', 'labels.json'):
            p = os.path.join(sdir, fn)
            if os.path.isfile(p): lpath = p; break
        if not lpath: continue
        d0 = os.path.join(DATA_PROCESSED, sname + '_device0_processed.csv')
        d1 = os.path.join(DATA_PROCESSED, sname + '_device1_processed.csv')
        if not (os.path.isfile(d0) and os.path.isfile(d1)): continue
        with open(lpath, encoding='utf-8') as f:
            data = json.load(f)
        segs = data.get('segments', data) if isinstance(data, dict) else data
        wbl = defaultdict(list)
        for seg in segs:
            canon = map_label(seg.get('label', ''))
            if canon is None: continue
            s, e = seg.get('start'), seg.get('end')
            if s is None: continue
            if e is None: e = s + 2.0
            wbl[canon].append((float(s), float(e)))
        for label, windows in sorted(wbl.items()):
            n = process_session_v10(d0, d1, sname + '/' + label, label, windows)
            if n > 0:
                print(f'  {sname}/{label:<20s} {"":>34s} {n:>4d} cycles')

X_all = np.array([cm.ravel() for cm in all_matrices])  # (N, 768)
labels_all = np.array(all_labels)

print('\nTotal: ' + str(len(X_all)) + ' cycles, ' + str(X_all.shape[1]) + 'D')
print('Label distribution:')
for lab, cnt in sorted(Counter(labels_all).items(), key=lambda x: -x[1]):
    print(f'  {lab:<20s}: {cnt:>5d}')

In [ ]:
# ── Cell 5: PCA (capped at 50) + t-SNE + UMAP ───────────────

scaler = StandardScaler()
X_scaled_768 = scaler.fit_transform(X_all)

# PCA capped at 50 components (matching V10.1)
n_comp = min(50, X_scaled_768.shape[1], X_scaled_768.shape[0] - 1)
pca = PCA(n_components=n_comp, svd_solver='full')
X_pca = pca.fit_transform(X_scaled_768)
cumvar = np.cumsum(pca.explained_variance_ratio_)
print('PCA: 768D -> ' + str(n_comp) + 'D')
print('  Variance captured: ' + str(round(cumvar[-1]*100, 1)) + '%')
print('  Top 10: ' + str(round(cumvar[min(9,n_comp-1)]*100, 1)) + '%')
print('  Top 20: ' + str(round(cumvar[min(19,n_comp-1)]*100, 1)) + '%')

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, n_comp+1), cumvar*100, 'o-', color='#7F77DD', markersize=3)
ax.axhline(95, color='red', ls='--', lw=1, label='95%')
ax.set_xlabel('PCA components'); ax.set_ylabel('Cumulative variance %')
ax.set_title('PCA variance (capped at 50 components)'); ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'fig_pca_variance.png'), dpi=150)
plt.show()

X_scaled = X_pca  # cluster on PCA space

print('\nRunning t-SNE on ' + str(n_comp) + 'D...')
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
X_tsne = tsne.fit_transform(X_scaled)
print('  t-SNE done')

print('Running UMAP on ' + str(n_comp) + 'D...')
reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42)
X_umap = reducer.fit_transform(X_scaled)
print('  UMAP done')

np.save(os.path.join(RESULTS_DIR, 'X_pca.npy'), X_pca)
np.save(os.path.join(RESULTS_DIR, 'X_tsne.npy'), X_tsne)
np.save(os.path.join(RESULTS_DIR, 'X_umap.npy'), X_umap)
np.save(os.path.join(RESULTS_DIR, 'labels_all.npy'), labels_all)
print('Saved')

In [ ]:
# ── Cell 6: K-means (n_init=20, max_iter=500) ────────────────

K_VALUES = [5, 8, 10, 11, 12, 13, 15]
kmeans_results = {}
inertias = []
silhouettes = []
labeled_mask = labels_all != 'unlabeled'

print('K-means on ' + str(X_scaled.shape[1]) + 'D PCA:')
for k in K_VALUES:
    km = KMeans(n_clusters=k, random_state=42, n_init=20, max_iter=500)
    cl = km.fit_predict(X_scaled)
    sil = silhouette_score(X_scaled, cl)
    inertias.append(km.inertia_)
    silhouettes.append(sil)
    kmeans_results[k] = cl
    if labeled_mask.sum() > 0:
        ari = adjusted_rand_score(labels_all[labeled_mask], cl[labeled_mask])
        nmi = normalized_mutual_info_score(labels_all[labeled_mask], cl[labeled_mask])
        print(f'  k={k:>2d}: silhouette={sil:.3f}, ARI={ari:.3f}, NMI={nmi:.3f}')
    else:
        print(f'  k={k:>2d}: silhouette={sil:.3f}')

best_k = K_VALUES[np.argmax(silhouettes)]
print('Best k by silhouette: ' + str(best_k) + ' (' + str(round(max(silhouettes),3)) + ')')

In [ ]:
# ── Cell 7: HDBSCAN ──────────────────────────────────────────

clusterer = hdbscan.HDBSCAN(min_cluster_size=15, min_samples=5, metric='euclidean')
hdb_labels = clusterer.fit_predict(X_scaled)
n_clusters_hdb = len(set(hdb_labels)) - (1 if -1 in hdb_labels else 0)
n_noise = (hdb_labels == -1).sum()
print('HDBSCAN on ' + str(X_scaled.shape[1]) + 'D: ' + str(n_clusters_hdb) + ' clusters, ' + str(n_noise) + ' noise (' + str(round(n_noise/len(hdb_labels)*100,1)) + '%)')

non_noise = hdb_labels != -1
if labeled_mask.sum() > 0 and (labeled_mask & non_noise).sum() > 10:
    mask = labeled_mask & non_noise
    ari = adjusted_rand_score(labels_all[mask], hdb_labels[mask])
    nmi = normalized_mutual_info_score(labels_all[mask], hdb_labels[mask])
    print(f'ARI={ari:.3f}, NMI={nmi:.3f} (labeled non-noise)')

In [ ]:
# ── Cell 8: Plot — embeddings by known labels ────────────────

LABEL_COLORS = {
    'underhand': '#5DCAA5', 'overhand': '#7F77DD', 'dragon_roll': '#E24B4A',
    'sneak_overhand': '#D85A30', 'sneak_underhand': '#EF9F27',
    'CW': '#3498db', 'CCW': '#9b59b6', 'unlabeled': '#DDDDDD',
}

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
for ax, X_2d, method in zip(axes, [X_tsne, X_umap], ['t-SNE', 'UMAP']):
    mask_unl = labels_all == 'unlabeled'
    ax.scatter(X_2d[mask_unl, 0], X_2d[mask_unl, 1], c='#DDDDDD', s=5, alpha=0.3, label='unlabeled')
    for label in sorted(set(labels_all)):
        if label == 'unlabeled': continue
        mask = labels_all == label
        ax.scatter(X_2d[mask, 0], X_2d[mask, 1], c=LABEL_COLORS.get(label, '#888'),
                   s=15, alpha=0.7, label=label)
    ax.set_title(method + ' colored by known labels', fontsize=13)
    ax.legend(fontsize=8, markerscale=2)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'fig_embeddings_by_label.png'), dpi=150)
plt.show()

In [ ]:
# ── Cell 9: Plot — elbow + silhouette ─────────────────────────

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(K_VALUES, inertias, 'o-', color='#7F77DD', lw=2)
ax1.set_xlabel('k'); ax1.set_ylabel('Inertia'); ax1.set_title('Elbow Method')
ax1.set_xticks(K_VALUES)
ax2.plot(K_VALUES, silhouettes, 's-', color='#5DCAA5', lw=2)
ax2.set_xlabel('k'); ax2.set_ylabel('Silhouette Score'); ax2.set_title('Silhouette vs k')
ax2.set_xticks(K_VALUES)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'fig_elbow_silhouette.png'), dpi=150)
plt.show()

In [ ]:
# ── Cell 10: Plot — t-SNE by K-means (grid) ──────────────────

fig, axes = plt.subplots(2, 4, figsize=(24, 12))
for ax, k in zip(axes.flat[:len(K_VALUES)], K_VALUES):
    cl = kmeans_results[k]
    ax.scatter(X_tsne[:, 0], X_tsne[:, 1], c=cl, cmap='tab20', s=5, alpha=0.5)
    sil_val = silhouettes[K_VALUES.index(k)]
    ax.set_title(f'k={k} (sil={sil_val:.3f})', fontsize=10)
if len(K_VALUES) < 8:
    axes.flat[len(K_VALUES)].axis('off')
plt.suptitle('t-SNE colored by K-means clusters', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'fig_tsne_kmeans_grid.png'), dpi=150)
plt.show()

In [ ]:
# ── Cell 11: Plot — HDBSCAN ───────────────────────────────────

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))
noise_mask = hdb_labels == -1
for ax, X_2d, method in zip([ax1, ax2], [X_tsne, X_umap], ['t-SNE', 'UMAP']):
    ax.scatter(X_2d[noise_mask, 0], X_2d[noise_mask, 1], c='#DDDDDD', s=3, alpha=0.3, label='noise')
    sc = ax.scatter(X_2d[~noise_mask, 0], X_2d[~noise_mask, 1],
                    c=hdb_labels[~noise_mask], cmap='tab20', s=10, alpha=0.6)
    ax.set_title(method + ' + HDBSCAN (' + str(n_clusters_hdb) + ' clusters)', fontsize=13)
    plt.colorbar(sc, ax=ax, label='Cluster')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'fig_hdbscan_embeddings.png'), dpi=150)
plt.show()

In [ ]:
# ── Cell 12: Cluster analysis ─────────────────────────────────

def analyze_clusters(cluster_labels, name):
    unique = sorted(set(cluster_labels))
    if -1 in unique: unique.remove(-1)
    print('\n' + name + ': ' + str(len(unique)) + ' clusters')
    print(f'{"Cluster":>8s}  {"Size":>6s}  {"Dominant":>20s}  {"Purity":>8s}  Composition')
    print('-' * 90)
    for c in unique:
        mask = cluster_labels == c
        size = mask.sum()
        lc = Counter(labels_all[mask])
        lc_known = {k: v for k, v in lc.items() if k != 'unlabeled'}
        if lc_known:
            dom = max(lc_known, key=lc_known.get)
            pur = lc_known[dom] / sum(lc_known.values())
        else:
            dom = 'unlabeled'; pur = 0.0
        comp = ', '.join(f'{l}={c}' for l, c in sorted(lc.items(), key=lambda x: -x[1])[:5])
        print(f'  C{c:>3d}    {size:>5d}  {dom:>20s}  {pur:>7.1%}  {comp}')
    total_known = sum(1 for l in labels_all if l != 'unlabeled')
    if total_known > 0:
        correct = sum(max(Counter({k:v for k,v in Counter(labels_all[cluster_labels==c]).items() if k!='unlabeled'}).values(), default=0) for c in unique)
        print('Overall purity: ' + str(round(correct/total_known, 3)))

for k in K_VALUES:
    analyze_clusters(kmeans_results[k], 'K-means k=' + str(k))

analyze_clusters(hdb_labels, 'HDBSCAN')

In [ ]:
# ── Cell 13: Save ─────────────────────────────────────────────

results_df = pd.DataFrame({
    'session': [session_names[sid] for sid in all_session_ids],
    'label': labels_all,
    'hdbscan_cluster': hdb_labels,
})
for k in K_VALUES:
    results_df['kmeans_k' + str(k)] = kmeans_results[k]
results_df.to_csv(os.path.join(RESULTS_DIR, 'cluster_assignments.csv'), index=False)

print('\nSUMMARY')
print('Total cycles: ' + str(len(X_all)))
print('Feature dim: ' + str(X_all.shape[1]))
n_labeled = sum(labels_all != 'unlabeled')
n_unlabeled = sum(labels_all == 'unlabeled')
print('Labeled: ' + str(n_labeled) + ', Unlabeled: ' + str(n_unlabeled))
print('\nK-means silhouette:')
for k, sil in zip(K_VALUES, silhouettes):
    marker = ' <-- best' if k == best_k else ''
    print(f'  k={k:>2d}: {sil:.4f}{marker}')
print('\nHDBSCAN: ' + str(n_clusters_hdb) + ' natural clusters')
print('\nSaved: ' + RESULTS_DIR + '/')